In [1]:
import pandas as pd
import os
import sys
GTE_DIR=os.environ["GTE_DIR"]
from glaciation_time_estimator.auxiliary_func.config_reader import read_config

In [2]:
class Cloud_data:
    def __init__(self, years, config_fp=None):
        self.years = years
        self.config=read_config(config_fp)
        self.cloud_fps = {year: os.path.join(self.config["postprocessing_output_dir"],"Final_results",f"{year}_all.parquet") for year in years}
        self.glac_fps = {year: os.path.join(self.config["postprocessing_output_dir"],"Final_results",f"{year}_glac_04.parquet") for year in years}
        self.loaded_key=None
        self.load_single_chunk(years[0])
    def load_single_chunk(self,key):
        if self.loaded_key != key:
            self.cloud_chunk = pd.read_parquet(self.cloud_fps[key])
            self.glac_chunk = pd.read_parquet(self.glac_fps[key])
            self.glac_cloud_chunk = self.glac_chunk.drop_duplicates(subset="Cloud_ID",keep="first")
            self.cloud_chunk['is_glaciating'] = self.cloud_chunk.index.isin(self.glac_cloud_chunk['Cloud_ID'])
            self.cloud_chunk["month"]= pd.to_datetime(self.cloud_chunk['track_start_time']).dt.month
            assert self.glac_cloud_chunk["Cloud_ID"].isin(self.cloud_chunk.index).all(),"The files don't correspont to each other"
            self.loaded_key = key
    def execute_analysis(self,analysis_func, years_to_analyze = None):
        result_list = []
        if years_to_analyze is None:
            years_to_analyze=self.years
        for year in years_to_analyze:
            self.load_single_chunk(year)
            result_list.append(analysis_func((self.cloud_chunk,self.glac_chunk, self.glac_cloud_chunk)))
        return result_list


In [3]:
dataset = Cloud_data([2007,2007,2008], config_fp=os.path.join(GTE_DIR,'configs/2007_tracking/01_01.yaml'))

In [4]:
dataset.cloud_chunk.keys()

Index(['is_large_pix_cloud', 'is_cot_valid_cloud', 'is_ctp_valid_cloud',
       'is_liq', 'is_mix', 'is_ice', 'max_water_frac', 'max_ice_fraction',
       'avg_size[km]', 'max_size[km]', 'min_size[km]', 'avg_size[px]',
       'max_size[px]', 'min_size[px]', 'track_start_time', 'track_length',
       'avg_cot', 'avg_ctp', 'glaciation_start_time', 'glaciation_end_time',
       'avg_lat', 'avg_lon', 'start_ice_fraction', 'end_ice_fraction',
       'ice_frac_hist', 'cot_hist', 'cot_nan_frac_hist', 'ctp_hist',
       'ctp_nan_frac_hist', 'lat_hist', 'lon_hist', 'size_hist_km', 'min_temp',
       'max_temp', 'pole', 'Hemisphere', 'Lifetime [h]', 'Radius [km]',
       'Level', 'Optical Thickness', 'Cloud type', 'Season', 'is_glaciating',
       'month'],
      dtype='object')

In [5]:
def mean_area(dfs):
    clouds,glac,glac_clouds = dfs
    return clouds["avg_size[km]"].mean()

In [6]:
dataset.execute_analysis(mean_area)

[6714.585403059159, 6714.585403059159, 6895.265018672163]